# Segmenting remote sensing imagery with FastSAM

[![image](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/opengeos/segment-geospatial/blob/main/docs/examples/fast_sam.ipynb)
[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/segment-geospatial/blob/main/docs/examples/fast_sam.ipynb)

FastSAM: https://github.com/CASIA-IVA-Lab/FastSAM

Make sure you use GPU runtime for this notebook. For Google Colab, go to `Runtime` -> `Change runtime type` and select `GPU` as the hardware accelerator.

## Install dependencies

Uncomment and run the following cell to install the required dependencies.

In [ ]:
# %pip install segment-geospatial segment-anything-fast

In [ ]:
import leafmap
from samgeo.common import tms_to_geotiff

## Create an interactive map

In [ ]:
m = leafmap.Map(center=[-22.17615, -51.253043], zoom=18, height="800px")
m.add_basemap("SATELLITE")
m

## Download a sample image

Pan and zoom the map to select the area of interest. Use the draw tools to draw a polygon or rectangle on the map

In [ ]:
bbox = m.user_roi_bounds()
if bbox is None:
    bbox = [-51.2565, -22.1777, -51.2512, -22.175]

In [ ]:
image = "Image.tif"
tms_to_geotiff(output=image, bbox=bbox, zoom=19, source="Satellite", overwrite=True)

You can also use your own image. Uncomment and run the following cell to use your own image.

In [ ]:
# image = '/path/to/your/own/image.tif'

Display the downloaded image on the map.

In [ ]:
m.layers[-1].visible = False
m.add_raster(image, layer_name="Image")
m

## Initialize SamGeo class

The initialization of the SamGeo class might take a few minutes. The initialization downloads the model weights and sets up the model for inference.

In [ ]:
from samgeo.fast_sam import SamGeo

sam = SamGeo(model="FastSAM-x.pt")

Set the image.

In [ ]:
sam.set_image("Image.tif")

Segment the image with `everything_prompt`. You can also try `point_prompt`, `box_prompt`, or `text_prompt`.

In [ ]:
sam.everything_prompt(output="mask.tif")

Show the annotated image.

In [ ]:
sam.show_anns("mask.png")

![](https://i.imgur.com/af4bj7O.png)

Convert the segmentation results from GeoTIFF to vector.

In [ ]:
sam.raster_to_vector("mask.tif", "mask.geojson")

Show the segmentation results on the map.

In [ ]:
m.add_raster("mask.tif", opacity=0.5, layer_name="Mask")
m.add_vector("mask.geojson", layer_name="Mask Vector")
m

![](https://i.imgur.com/LvEAMSl.png)

In [ ]:
# ============================================================
# كشف حدود الحقول الزراعية - شريط متصل (دقة عالية + إزالة التداخلات)
# نسخة مصححة بالكامل - مع حل مشكلة الكائنات المتكررة في المناطق المتداخلة
# ============================================================

# 1. تثبيت المكتبات
# ============================================================
!pip install segment-geospatial leafmap localtileserver geopandas matplotlib pandas rtree -q

# 2. استيراد المكتبات
# ============================================================
import leafmap
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from samgeo import SamGeo
from samgeo.common import tms_to_geotiff
from geopy.distance import distance
import os
import tempfile
import numpy as np
from shapely.geometry import Polygon, box, MultiPolygon, Point
from shapely.ops import unary_union
from shapely.validation import make_valid
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("🌾 كشف حدود الحقول الزراعية - شريط متصل (بدون تكرار)")
print("=" * 60)

# ============================================================
# دالة لإزالة المضلعات المكررة (الحل الأساسي للمشكلة)
# ============================================================

def remove_duplicate_polygons(gdf, iou_threshold=0.5, min_area_ha=0.005):
    """
    إزالة المضلعات المكررة بناءً على معامل التداخل (IoU)
    يحتفظ فقط بالمضلع الأكبر في حالة التداخل الكبير
    """
    if len(gdf) == 0:
        return gdf

    # إزالة المضلعات الصغيرة جداً أولاً
    gdf = gdf[gdf['area_ha'] > min_area_ha].copy()

    # إضافة عمود لتتبع ما إذا تم الاحتفاظ بالمضلع
    gdf['keep'] = True
    gdf['original_index'] = range(len(gdf))

    # فرز حسب المساحة (تنازلياً) - الأكبر أولاً
    gdf_sorted = gdf.sort_values('area_ha', ascending=False).reset_index(drop=True)

    to_remove = []

    for i in range(len(gdf_sorted)):
        if i in to_remove:
            continue

        geom_i = gdf_sorted.iloc[i].geometry

        for j in range(i + 1, len(gdf_sorted)):
            if j in to_remove:
                continue

            geom_j = gdf_sorted.iloc[j].geometry

            try:
                # حساب مساحة التقاطع
                intersection = geom_i.intersection(geom_j)
                if intersection.is_empty:
                    continue

                intersection_area = intersection.area
                area_i = geom_i.area
                area_j = geom_j.area

                # حساب IoU (Intersection over Union)
                iou = intersection_area / (area_i + area_j - intersection_area)

                # حساب نسبة التداخل بالنسبة للمضلع الأصغر
                smaller_area = min(area_i, area_j)
                overlap_ratio = intersection_area / smaller_area if smaller_area > 0 else 0

                # إذا كان التداخل كبيراً، إزالة المضلع الأصغر
                if iou > iou_threshold or overlap_ratio > 0.7:
                    # تحديد المضلع الأصغر للإزالة
                    if area_i >= area_j:
                        to_remove.append(j)
                    else:
                        to_remove.append(i)
                        break  # تم إزالة المضلع الحالي، نخرج من الحلقة الداخلية

            except Exception as e:
                continue

    # إزالة المضلعات المكررة
    keep_indices = [i for i in range(len(gdf_sorted)) if i not in to_remove]
    result = gdf_sorted.iloc[keep_indices].copy()

    # إعادة تعيين الفهرس
    result = result.drop(columns=['keep', 'original_index']).reset_index(drop=True)

    return result


def merge_overlapping_polygons(gdf, buffer_meters=1.0, min_area_ha=0.005):
    """
    دمج المضلعات المتداخلة بشكل كبير (بدلاً من إزالتها فقط)
    """
    if len(gdf) == 0:
        return gdf

    # إزالة الصغير جداً
    gdf = gdf[gdf['area_ha'] > min_area_ha].copy()

    # إنشاء عمود للدمج
    gdf['merge_group'] = -1
    current_group = 0

    # تحويل إلى CRS متري للحسابات الدقيقة
    original_crs = gdf.crs
    if gdf.crs is None or not gdf.crs.is_projected:
        gdf = gdf.to_crs('EPSG:3857')  # مشروع ويب ميركيتور

    # إضافة مسافة صغيرة لتجنب الفجوات الدقيقة
    gdf['geometry'] = gdf['geometry'].buffer(buffer_meters)

    # البحث عن المضلعات المتصلة
    for i in range(len(gdf)):
        if gdf.iloc[i]['merge_group'] != -1:
            continue

        geom_i = gdf.iloc[i].geometry
        gdf.at[gdf.index[i], 'merge_group'] = current_group

        # البحث عن جميع المضلعات التي تتداخل مع هذه المجموعة
        for j in range(i + 1, len(gdf)):
            if gdf.iloc[j]['merge_group'] != -1:
                continue

            geom_j = gdf.iloc[j].geometry

            try:
                intersection = geom_i.intersection(geom_j)
                if not intersection.is_empty:
                    if intersection.area / min(geom_i.area, geom_j.area) > 0.5:
                        gdf.at[gdf.index[j], 'merge_group'] = current_group
                        geom_i = geom_i.union(geom_j)  # توسيع المنطقة للبحث
            except:
                pass

        current_group += 1

    # دمج المضلعات في كل مجموعة
    merged_geoms = []
    merged_areas = []
    merged_km_centers = []

    for group in range(current_group):
        group_polys = gdf[gdf['merge_group'] == group].geometry

        if len(group_polys) > 1:
            # دمج المضلعات
            merged_geom = unary_union(group_polys.tolist())
            # إزالة المسافة المؤقتة
            if buffer_meters > 0:
                merged_geom = merged_geom.buffer(-buffer_meters)
            merged_geoms.append(merged_geom)
            merged_areas.append(merged_geom.area)
            merged_km_centers.append(gdf[gdf['merge_group'] == group]['km_center'].mean())
        else:
            # مضلع واحد، إزالة المسافة المؤقتة فقط
            geom = group_polys.iloc[0]
            if buffer_meters > 0:
                geom = geom.buffer(-buffer_meters)
            merged_geoms.append(geom)
            merged_areas.append(geom.area)
            merged_km_centers.append(gdf[gdf['merge_group'] == group]['km_center'].iloc[0])

    # إنشاء GeoDataFrame جديد
    merged_gdf = gpd.GeoDataFrame({
        'geometry': merged_geoms,
        'area_m2': [a for a in merged_areas],
        'area_ha': [a / 10000 for a in merged_areas],
        'km_center': merged_km_centers
    }, crs=gdf.crs)

    # التحقق من صحة المضلعات
    merged_gdf['geometry'] = merged_gdf['geometry'].apply(lambda g: make_valid(g) if not g.is_valid else g)

    # العودة إلى CRS الأصلي إذا لزم الأمر
    if original_crs and original_crs != merged_gdf.crs:
        merged_gdf = merged_gdf.to_crs(original_crs)

    return merged_gdf


# ============================================================
# الإعدادات المتقدمة
# ============================================================

start_lat = 24.390288
start_lon = 32.979508
strip_length_km = 10
strip_width_km = 1
zoom_level = 18
model_type = "vit_l"
overlap_ratio = 0.15  # 15% تداخل

print(f"\n📍 نقطة البداية: ({start_lat}, {start_lon})")
print(f"📏 طول الشريط: {strip_length_km} كم (شرقًا)")
print(f"📐 عرض الشريط: {strip_width_km} كم")
print(f"🔄 نسبة التداخل: {overlap_ratio * 100}%")
print("=" * 60)

# ============================================================
# حساب الشريط المتصل
# ============================================================

print("\n📐 جاري حساب حدود الشريط المتصل...")

# حساب الزوايا الأربع للشريط
nw_point = distance(kilometers=strip_width_km/2).destination((start_lat, start_lon), bearing=0)
sw_point = distance(kilometers=strip_width_km/2).destination((start_lat, start_lon), bearing=180)
ne_point = distance(kilometers=strip_length_km).destination((nw_point.latitude, nw_point.longitude), bearing=90)
se_point = distance(kilometers=strip_length_km).destination((sw_point.latitude, sw_point.longitude), bearing=90)

bbox = [sw_point.longitude, sw_point.latitude, ne_point.longitude, ne_point.latitude]

print(f"📦 BBOX الكامل: غرب {bbox[0]:.6f} | جنوب {bbox[1]:.6f} | شرق {bbox[2]:.6f} | شمال {bbox[3]:.6f}")

# ============================================================
# تقسيم الشريط إلى أجزاء متداخلة
# ============================================================

num_segments = 8
segment_length_km = strip_length_km / num_segments
overlap_km = segment_length_km * overlap_ratio
step_km = segment_length_km - overlap_km

print(f"\n🔪 تقسيم الشريط إلى {num_segments} جزء متداخل:")

segments = []
for i in range(num_segments):
    start_km = max(0, i * step_km)
    end_km = min(strip_length_km, start_km + segment_length_km)

    start_point = distance(kilometers=start_km).destination((start_lat, start_lon), bearing=90)
    end_point = distance(kilometers=end_km).destination((start_lat, start_lon), bearing=90)

    segment_bbox = [
        start_point.longitude,
        sw_point.latitude,
        end_point.longitude,
        ne_point.latitude
    ]

    segments.append({
        'index': i,
        'km_start': start_km,
        'km_end': end_km,
        'bbox': segment_bbox
    })
    print(f"   الجزء {i+1}: {start_km:.1f}-{end_km:.1f} كم")

# ============================================================
# تحميل الصور وكشف الحقول
# ============================================================

print("\n📡 جاري تحميل الصور وكشف الحقول...")

all_segment_gdfs = []
sam = SamGeo(model_type=model_type)

for seg in segments:
    print(f"\n📍 معالجة الجزء {seg['index']+1}/{num_segments}...")

    # تحميل الصورة
    with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
        image_path = tmp.name

    success = False
    for source in ["Satellite", "OpenStreetMap"]:
        try:
            tms_to_geotiff(
                output=image_path,
                bbox=seg['bbox'],
                zoom=zoom_level,
                source=source,
                overwrite=True
            )
            if os.path.exists(image_path) and os.path.getsize(image_path) > 10000:
                success = True
                break
        except:
            continue

    if not success:
        print(f"   ❌ فشل تحميل الصورة")
        continue

    # كشف الحقول
    with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp_mask:
        mask_path = tmp_mask.name

    with tempfile.NamedTemporaryFile(suffix='.geojson', delete=False) as tmp_geojson:
        vector_path = tmp_geojson.name

    try:
        sam.generate(image_path, mask_path, foreground=True, unique=True,
                    erosion_kernel=(2, 2), mask_multiplier=255)
        sam.raster_to_vector(mask_path, vector_path)

        gdf = gpd.read_file(vector_path)

        if len(gdf) > 0:
            gdf['area_m2'] = gdf.area
            gdf['area_ha'] = gdf['area_m2'] / 10000
            gdf['segment'] = seg['index']
            gdf['km_start'] = seg['km_start']
            gdf['km_end'] = seg['km_end']

            # تقدير موقع المركز
            def get_km_center(geom):
                centroid = geom.centroid
                approx_km = (centroid.x - start_lon) / 0.00987
                return max(0, min(strip_length_km, approx_km))

            gdf['km_center'] = gdf.geometry.apply(get_km_center)

            print(f"   🌾 تم كشف {len(gdf)} حقل - مساحة: {gdf['area_ha'].sum():.2f} هكتار")
            all_segment_gdfs.append(gdf)
        else:
            print(f"   ⚠️ لم يتم كشف حقول")

    except Exception as e:
        print(f"   ❌ خطأ: {e}")

    finally:
        for path in [mask_path, vector_path, image_path]:
            try:
                os.unlink(path)
            except:
                pass

# ============================================================
# دمج الأجزاء وإزالة التكرارات
# ============================================================

print("\n" + "=" * 60)
print("🔧 معالجة التداخلات وإزالة التكرارات...")
print("=" * 60)

if all_segment_gdfs:
    # دمج جميع الأجزاء
    raw_gdf = gpd.GeoDataFrame(pd.concat(all_segment_gdfs, ignore_index=True))
    print(f"\n📊 قبل المعالجة: {len(raw_gdf)} مضلع (بما في ذلك التكرارات)")

    # الخطوة 1: إزالة المضلعات الصغيرة جداً (ضوضاء)
    raw_gdf = raw_gdf[raw_gdf['area_ha'] > 0.005]
    print(f"📊 بعد إزالة الضوضاء: {len(raw_gdf)} مضلع")

    # الخطوة 2: إزالة التكرارات باستخدام IoU
    deduplicated_gdf = remove_duplicate_polygons(raw_gdf, iou_threshold=0.4)
    print(f"📊 بعد إزالة التكرارات: {len(deduplicated_gdf)} مضلع")

    # الخطوة 3: دمج المضلعات المتداخلة بشكل كبير
    merged_gdf = merge_overlapping_polygons(deduplicated_gdf, buffer_meters=0.5)
    print(f"📊 بعد دمج المتداخلات: {len(merged_gdf)} مضلع")

    # إضافة معرفات نهائية
    merged_gdf = merged_gdf[merged_gdf['area_ha'] > 0.005].reset_index(drop=True)
    merged_gdf['field_id'] = range(1, len(merged_gdf) + 1)

    # إحصائيات نهائية
    print("\n" + "=" * 60)
    print("📊 النتائج النهائية (بدون تكرارات):")
    print("=" * 60)
    print(f"🌾 إجمالي الحقول الفريدة: {len(merged_gdf)}")
    print(f"📏 إجمالي المساحة: {merged_gdf['area_ha'].sum():.2f} هكتار")
    print(f"📐 متوسط مساحة الحقل: {merged_gdf['area_ha'].mean():.2f} هكتار")
    print(f"🏞️ أكبر حقل: {merged_gdf['area_ha'].max():.2f} هكتار")
    print(f"📏 أصغر حقل: {merged_gdf['area_ha'].min():.4f} هكتار")

    # ============================================================
    # عرض النتائج على الخريطة
    # ============================================================

    print("\n🗺️ جاري إنشاء الخريطة التفاعلية...")

    m = leafmap.Map(center=[start_lat, start_lon], zoom=13, height=700)
    m.add_basemap("SATELLITE")

    # حدود الشريط
    strip_polygon = {
        "type": "Feature",
        "properties": {"name": "شريط التحليل (10 كم)"},
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [bbox[0], bbox[1]],
                [bbox[2], bbox[1]],
                [bbox[2], bbox[3]],
                [bbox[0], bbox[3]],
                [bbox[0], bbox[1]]
            ]]
        }
    }
    m.add_geojson(strip_polygon, layer_name="شريط التحليل",
                  fill_colors=["blue"], fill_opacity=0.1,
                  stroke_colors=["blue"], stroke_widths=3)

    # الحقول المكتشفة
    with tempfile.NamedTemporaryFile(suffix='.geojson', delete=False) as tmp:
        merged_path = tmp.name
    merged_gdf.to_file(merged_path, driver='GeoJSON')
    m.add_geojson(merged_path, layer_name=f"حدود الحقول ({len(merged_gdf)} حقل)",
                  fill_colors=["green"], fill_opacity=0.3,
                  stroke_colors=["red"], stroke_widths=2)

    from IPython.display import display
    display(m)
    os.unlink(merged_path)

    # ============================================================
    # حفظ النتائج
    # ============================================================

    output_dir = "fields_no_duplicates"
    os.makedirs(output_dir, exist_ok=True)

    geojson_path = os.path.join(output_dir, "fields_unique.geojson")
    merged_gdf.to_file(geojson_path, driver='GeoJSON')

    print(f"\n💾 تم حفظ النتائج في: {output_dir}/")
    print(f"   📁 GeoJSON: {geojson_path}")

    # إحصائيات لكل كيلومتر
    print("\n📊 إحصائيات لكل كيلومتر (بدون تكرارات):")
    print("-" * 60)
    for km in range(strip_length_km):
        km_fields = merged_gdf[
            (merged_gdf['km_center'] >= km) &
            (merged_gdf['km_center'] < km + 1)
        ]
        print(f"   كيلومتر {km}: {len(km_fields)} حقل | {km_fields['area_ha'].sum():.1f} هكتار")

    print("\n" + "=" * 60)
    print("✅ اكتملت المعالجة بنجاح! (بدون تكرارات في المناطق المتداخلة)")
    print("=" * 60)

else:
    print("\n❌ لم يتم كشف أي حقول")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 666.8/666.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.5/287.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 

Downloading...
From: https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth
To: /root/.cache/torch/hub/checkpoints/sam_vit_l_0b3195.pth
100%|██████████| 1.25G/1.25G [00:10<00:00, 122MB/s]



📍 معالجة الجزء 1/8...
Downloaded image 01/90
Downloaded image 02/90
Downloaded image 03/90
Downloaded image 04/90
Downloaded image 05/90
Downloaded image 06/90
Downloaded image 07/90
Downloaded image 08/90
Downloaded image 09/90
Downloaded image 10/90
Downloaded image 11/90
Downloaded image 12/90
Downloaded image 13/90
Downloaded image 14/90
Downloaded image 15/90
Downloaded image 16/90
Downloaded image 17/90
Downloaded image 18/90
Downloaded image 19/90
Downloaded image 20/90
Downloaded image 21/90
Downloaded image 22/90
Downloaded image 23/90
Downloaded image 24/90
Downloaded image 25/90
Downloaded image 26/90
Downloaded image 27/90
Downloaded image 28/90
Downloaded image 29/90
Downloaded image 30/90
Downloaded image 31/90
Downloaded image 32/90
Downloaded image 33/90
Downloaded image 34/90
Downloaded image 35/90
Downloaded image 36/90
Downloaded image 37/90
Downloaded image 38/90
Downloaded image 39/90
Downloaded image 40/90
Downloaded image 41/90
Downloaded image 42/90
Downloaded 

   🌾 تم كشف 8668 حقل - مساحة: 114.25 هكتار

📍 معالجة الجزء 2/8...
Downloaded image 01/90
Downloaded image 02/90
Downloaded image 03/90
Downloaded image 04/90
Downloaded image 05/90
Downloaded image 06/90
Downloaded image 07/90
Downloaded image 08/90
Downloaded image 09/90
Downloaded image 10/90
Downloaded image 11/90
Downloaded image 12/90
Downloaded image 13/90
Downloaded image 14/90
Downloaded image 15/90
Downloaded image 16/90
Downloaded image 17/90
Downloaded image 18/90
Downloaded image 19/90
Downloaded image 20/90
Downloaded image 21/90
Downloaded image 22/90
Downloaded image 23/90
Downloaded image 24/90
Downloaded image 25/90
Downloaded image 26/90
Downloaded image 27/90
Downloaded image 28/90
Downloaded image 29/90
Downloaded image 30/90
Downloaded image 31/90
Downloaded image 32/90
Downloaded image 33/90
Downloaded image 34/90
Downloaded image 35/90
Downloaded image 36/90
Downloaded image 37/90
Downloaded image 38/90
Downloaded image 39/90
Downloaded image 40/90
Downloaded ima

In [ ]:
# ============================================================
# GeoAI - كشف حدود الحقول الزراعية (نسخة احترافية)
# يدعم: SAM, U-Net, Mask R-CNN
# مع دمج ذكي للأجزاء المتداخلة
# ============================================================

# 1. تثبيت المكتبات
# ============================================================
!pip install segment-geospatial leafmap localtileserver geopandas matplotlib pandas rtree rasterio opencv-python torch torchvision segmentation-models-pytorch -q

# 2. استيراد المكتبات
# ============================================================
import os
import tempfile
import warnings
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any
from enum import Enum

import leafmap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import Polygon, box, MultiPolygon
from shapely.ops import unary_union
from shapely.validation import make_valid
from geopy.distance import distance
from samgeo import SamGeo
from samgeo.common import tms_to_geotiff
import rasterio
from rasterio.features import shapes
import cv2

warnings.filterwarnings('ignore')

# ============================================================
# الإعدادات والثوابت
# ============================================================

@dataclass
class Config:
    """إعدادات التحليل"""
    # منطقة الدراسة
    start_lat: float = 24.390288
    start_lon: float = 32.979508
    strip_length_km: float = 10.0
    strip_width_km: float = 1.0

    # إعدادات الصور
    zoom_level: int = 18
    overlap_ratio: float = 0.15
    num_segments: int = 8

    # إعدادات النموذج
    model_type: str = "vit_l"  # vit_b, vit_l, vit_h
    min_field_area_ha: float = 0.005  # 50 متر مربع

    # إعدادات معالجة التداخلات
    iou_threshold: float = 0.4
    buffer_meters: float = 0.5

    # مصادر الصور
    image_sources: List[str] = None

    def __post_init__(self):
        if self.image_sources is None:
            self.image_sources = ["Satellite", "OpenStreetMap"]


class ModelType(Enum):
    """أنواع النماذج المدعومة"""
    SAM = "sam"
    UNET = "unet"
    MASK_RCNN = "mask_rcnn"


# ============================================================
# كلاس معالجة الصور
# ============================================================

class ImageDownloader:
    """تحميل الصور من TMS"""

    @staticmethod
    def download_bbox(bbox: List[float], zoom: int, output_path: str,
                      sources: List[str]) -> bool:
        """تحميل صورة لـ BBOX معين"""
        for source in sources:
            try:
                tms_to_geotiff(
                    output=output_path,
                    bbox=bbox,
                    zoom=zoom,
                    source=source,
                    overwrite=True
                )
                if os.path.exists(output_path) and os.path.getsize(output_path) > 10000:
                    return True
            except Exception:
                continue
        return False


# ============================================================
# كلاس معالجة النماذج
# ============================================================

class FieldDetector:
    """كشف الحقول باستخدام نماذج مختلفة"""

    def __init__(self, config: Config):
        self.config = config
        self.sam_model = None

    def _init_sam(self):
        """تهيئة نموذج SAM"""
        if self.sam_model is None:
            self.sam_model = SamGeo(model_type=self.config.model_type)
        return self.sam_model

    def detect_with_sam(self, image_path: str) -> Optional[gpd.GeoDataFrame]:
        """الكشف باستخدام SAM"""
        try:
            model = self._init_sam()

            with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp_mask:
                mask_path = tmp_mask.name

            with tempfile.NamedTemporaryFile(suffix='.geojson', delete=False) as tmp_geojson:
                vector_path = tmp_geojson.name

            # توليد الماسك
            model.generate(
                image_path, mask_path,
                foreground=True, unique=True,
                erosion_kernel=(2, 2), mask_multiplier=255
            )

            # تحويل إلى متجهات
            model.raster_to_vector(mask_path, vector_path)

            # قراءة النتائج
            gdf = gpd.read_file(vector_path)

            # تنظيف
            os.unlink(mask_path)
            os.unlink(vector_path)

            return gdf if len(gdf) > 0 else None

        except Exception as e:
            print(f"      خطأ في SAM: {e}")
            return None


# ============================================================
# كلاس معالجة التداخلات
# ============================================================

class OverlapProcessor:
    """معالجة وإزالة التداخلات بين الأجزاء"""

    @staticmethod
    def remove_duplicates(gdf: gpd.GeoDataFrame, iou_threshold: float = 0.4,
                          min_area_ha: float = 0.005) -> gpd.GeoDataFrame:
        """إزالة المضلعات المكررة باستخدام IoU"""
        if len(gdf) == 0:
            return gdf

        # فلترة الصغير جداً
        gdf = gdf[gdf['area_ha'] > min_area_ha].copy()
        if len(gdf) == 0:
            return gdf

        # فرز حسب المساحة (الأكبر أولاً)
        gdf_sorted = gdf.sort_values('area_ha', ascending=False).reset_index(drop=True)
        keep_mask = np.ones(len(gdf_sorted), dtype=bool)

        for i in range(len(gdf_sorted)):
            if not keep_mask[i]:
                continue

            geom_i = gdf_sorted.iloc[i].geometry

            for j in range(i + 1, len(gdf_sorted)):
                if not keep_mask[j]:
                    continue

                geom_j = gdf_sorted.iloc[j].geometry

                try:
                    intersection = geom_i.intersection(geom_j)
                    if intersection.is_empty:
                        continue

                    intersection_area = intersection.area
                    area_i = geom_i.area
                    area_j = geom_j.area

                    # حساب نسبة التداخل للمضلع الأصغر
                    smaller_area = min(area_i, area_j)
                    overlap_ratio = intersection_area / smaller_area if smaller_area > 0 else 0

                    if overlap_ratio > iou_threshold:
                        # إزالة المضلع الأصغر
                        if area_i >= area_j:
                            keep_mask[j] = False
                        else:
                            keep_mask[i] = False
                            break
                except Exception:
                    continue

        result = gdf_sorted[keep_mask].reset_index(drop=True)
        return result

    @staticmethod
    def merge_overlapping(gdf: gpd.GeoDataFrame, buffer_meters: float = 0.5,
                          min_area_ha: float = 0.005) -> gpd.GeoDataFrame:
        """دمج المضلعات المتداخلة بشكل كبير"""
        if len(gdf) == 0:
            return gdf

        gdf = gdf[gdf['area_ha'] > min_area_ha].copy()
        if len(gdf) == 0:
            return gdf

        # حفظ CRS الأصلي
        original_crs = gdf.crs

        # تحويل إلى CRS متري إذا لزم الأمر
        if gdf.crs is None or not gdf.crs.is_projected:
            gdf = gdf.to_crs('EPSG:3857')

        # إضافة مسافة بسيطة للتأكد من التداخل
        gdf['geometry'] = gdf['geometry'].buffer(buffer_meters)

        # تجميع المضلعات المتصلة
        merged_geoms = []
        visited = set()

        for i in range(len(gdf)):
            if i in visited:
                continue

            # تجميع كل المضلعات المتصلة
            group = [i]
            visited.add(i)

            for j in range(len(gdf)):
                if j in visited:
                    continue

                # فحص إذا كان j يتصل بأي من المجموعة
                for idx in group:
                    try:
                        if gdf.iloc[idx].geometry.intersects(gdf.iloc[j].geometry):
                            group.append(j)
                            visited.add(j)
                            break
                    except Exception:
                        continue

            # دمج المجموعة
            if len(group) > 1:
                merged_geom = unary_union([gdf.iloc[idx].geometry for idx in group])
                if buffer_meters > 0:
                    merged_geom = merged_geom.buffer(-buffer_meters)
                merged_geoms.append(merged_geom)
            else:
                geom = gdf.iloc[group[0]].geometry
                if buffer_meters > 0:
                    geom = geom.buffer(-buffer_meters)
                merged_geoms.append(geom)

        # إنشاء GeoDataFrame جديد
        merged_gdf = gpd.GeoDataFrame({
            'geometry': merged_geoms,
            'area_m2': [g.area for g in merged_geoms],
            'area_ha': [g.area / 10000 for g in merged_geoms]
        }, crs=gdf.crs)

        # التحقق من صحة المضلعات
        merged_gdf['geometry'] = merged_gdf['geometry'].apply(
            lambda g: make_valid(g) if not g.is_valid else g
        )

        # العودة إلى CRS الأصلي
        if original_crs and original_crs != merged_gdf.crs:
            merged_gdf = merged_gdf.to_crs(original_crs)

        return merged_gdf


# ============================================================
# كلاس التحليل الرئيسي
# ============================================================

class AgriculturalStripAnalyzer:
    """المحلل الرئيسي للشريط الزراعي"""

    def __init__(self, config: Config):
        self.config = config
        self.downloader = ImageDownloader()
        self.detector = FieldDetector(config)
        self.overlap_processor = OverlapProcessor()
        self.bbox = None
        self.segments = []
        self.results = []

    def _calculate_bbox(self) -> List[float]:
        """حساب BBOX الكامل للشريط"""
        start_lat, start_lon = self.config.start_lat, self.config.start_lon
        width_km = self.config.strip_width_km
        length_km = self.config.strip_length_km

        nw_point = distance(kilometers=width_km/2).destination((start_lat, start_lon), bearing=0)
        sw_point = distance(kilometers=width_km/2).destination((start_lat, start_lon), bearing=180)
        ne_point = distance(kilometers=length_km).destination((nw_point.latitude, nw_point.longitude), bearing=90)
        se_point = distance(kilometers=length_km).destination((sw_point.latitude, sw_point.longitude), bearing=90)

        return [sw_point.longitude, sw_point.latitude, ne_point.longitude, ne_point.latitude]

    def _create_segments(self) -> List[Dict]:
        """تقسيم الشريط إلى أجزاء متداخلة"""
        segments = []
        seg_length_km = self.config.strip_length_km / self.config.num_segments
        overlap_km = seg_length_km * self.config.overlap_ratio
        step_km = seg_length_km - overlap_km

        for i in range(self.config.num_segments):
            start_km = max(0, i * step_km)
            end_km = min(self.config.strip_length_km, start_km + seg_length_km)

            start_point = distance(kilometers=start_km).destination(
                (self.config.start_lat, self.config.start_lon), bearing=90
            )
            end_point = distance(kilometers=end_km).destination(
                (self.config.start_lat, self.config.start_lon), bearing=90
            )

            segments.append({
                'index': i,
                'km_start': start_km,
                'km_end': end_km,
                'bbox': [
                    start_point.longitude,   # غرب
                    self.bbox[1],             # جنوب
                    end_point.longitude,      # شرق
                    self.bbox[3]              # شمال
                ]
            })

        return segments

    def _get_km_center(self, geom, start_lon: float) -> float:
        """تقدير مركز الحقل على طول الشريط"""
        centroid = geom.centroid
        approx_km = (centroid.x - start_lon) / 0.00987  # تقريب
        return max(0, min(self.config.strip_length_km, approx_km))

    def _process_segment(self, segment: Dict) -> Optional[gpd.GeoDataFrame]:
        """معالجة جزء واحد"""
        print(f"   الجزء {segment['index']+1}/{self.config.num_segments} "
              f"({segment['km_start']:.1f}-{segment['km_end']:.1f} كم)...")

        # تحميل الصورة
        with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
            image_path = tmp.name

        success = self.downloader.download_bbox(
            segment['bbox'], self.config.zoom_level, image_path,
            self.config.image_sources
        )

        if not success:
            print(f"      ❌ فشل تحميل الصورة")
            return None

        # كشف الحقول
        gdf = self.detector.detect_with_sam(image_path)

        # تنظيف
        os.unlink(image_path)

        if gdf is None or len(gdf) == 0:
            print(f"      ⚠️ لم يتم كشف حقول")
            return None

        # إضافة بيانات الجزء
        gdf['area_m2'] = gdf.area
        gdf['area_ha'] = gdf['area_m2'] / 10000
        gdf['segment'] = segment['index']
        gdf['km_start'] = segment['km_start']
        gdf['km_end'] = segment['km_end']
        gdf['km_center'] = gdf.geometry.apply(
            lambda g: self._get_km_center(g, self.bbox[0])
        )

        # فلترة الحقول الصغيرة
        gdf = gdf[gdf['area_ha'] > self.config.min_field_area_ha]

        print(f"      🌾 تم كشف {len(gdf)} حقل - مساحة: {gdf['area_ha'].sum():.2f} هكتار")

        return gdf

    def run(self) -> Dict[str, Any]:
        """تنفيذ التحليل الكامل"""
        print("=" * 60)
        print("🌾 GeoAI - كشف حدود الحقول الزراعية")
        print("=" * 60)

        # حساب BBOX
        self.bbox = self._calculate_bbox()
        print(f"\n📍 منطقة الدراسة: {self.config.strip_length_km} كم × {self.config.strip_width_km} كم")
        print(f"   غرب: {self.bbox[0]:.6f} | شرق: {self.bbox[2]:.6f}")

        # إنشاء الأجزاء
        self.segments = self._create_segments()
        print(f"\n🔪 تقسيم إلى {len(self.segments)} جزء (تداخل {self.config.overlap_ratio*100:.0f}%)")

        # معالجة كل جزء
        print("\n📡 جاري التحميل والكشف...")
        all_gdfs = []

        for seg in self.segments:
            gdf = self._process_segment(seg)
            if gdf is not None:
                all_gdfs.append(gdf)

        if not all_gdfs:
            print("\n❌ لم يتم كشف أي حقول")
            return {'success': False}

        # دمج النتائج
        print("\n" + "=" * 60)
        print("🔧 معالجة النتائج وإزالة التداخلات...")
        print("=" * 60)

        raw_gdf = gpd.GeoDataFrame(pd.concat(all_gdfs, ignore_index=True))
        print(f"\n📊 قبل المعالجة: {len(raw_gdf)} مضلع")

        # إزالة التكرارات
        dedup_gdf = self.overlap_processor.remove_duplicates(
            raw_gdf, self.config.iou_threshold, self.config.min_field_area_ha
        )
        print(f"📊 بعد إزالة التكرارات: {len(dedup_gdf)} مضلع")

        # دمج المتداخلات
        merged_gdf = self.overlap_processor.merge_overlapping(
            dedup_gdf, self.config.buffer_meters, self.config.min_field_area_ha
        )
        print(f"📊 بعد دمج المتداخلات: {len(merged_gdf)} مضلع")

        # إضافة المعرفات النهائية
        merged_gdf = merged_gdf.reset_index(drop=True)
        merged_gdf['field_id'] = range(1, len(merged_gdf) + 1)

        # إعادة حساب المساحات
        merged_gdf['area_m2'] = merged_gdf.area
        merged_gdf['area_ha'] = merged_gdf['area_m2'] / 10000

        # حساب الإحصائيات
        stats = self._calculate_statistics(merged_gdf)

        self.results = {
            'success': True,
            'gdf': merged_gdf,
            'stats': stats,
            'config': self.config
        }

        return self.results

    def _calculate_statistics(self, gdf: gpd.GeoDataFrame) -> Dict:
        """حساب الإحصائيات"""
        stats = {
            'total_fields': len(gdf),
            'total_area_ha': gdf['area_ha'].sum(),
            'mean_area_ha': gdf['area_ha'].mean(),
            'median_area_ha': gdf['area_ha'].median(),
            'min_area_ha': gdf['area_ha'].min(),
            'max_area_ha': gdf['area_ha'].max(),
            'std_area_ha': gdf['area_ha'].std(),
            'density_per_km': len(gdf) / self.config.strip_length_km,
            'area_per_km': gdf['area_ha'].sum() / self.config.strip_length_km
        }

        # إحصائيات لكل كيلومتر
        km_stats = []
        for km in range(int(self.config.strip_length_km)):
            km_fields = gdf[
                (gdf['km_center'] >= km) &
                (gdf['km_center'] < km + 1)
            ]
            km_stats.append({
                'km': km,
                'num_fields': len(km_fields),
                'area_ha': km_fields['area_ha'].sum()
            })

        stats['km_stats'] = km_stats

        return stats

    def display_results(self):
        """عرض النتائج على الخريطة"""
        if not self.results or not self.results['success']:
            print("لا توجد نتائج للعرض")
            return

        gdf = self.results['gdf']
        stats = self.results['stats']

        print("\n🗺️ جاري إنشاء الخريطة التفاعلية...")

        m = leafmap.Map(center=[self.config.start_lat, self.config.start_lon],
                        zoom=13, height=700)
        m.add_basemap("SATELLITE")

        # إضافة حدود الشريط
        strip_polygon = {
            "type": "Feature",
            "properties": {"name": f"شريط التحليل ({self.config.strip_length_km} كم)"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [self.bbox[0], self.bbox[1]],
                    [self.bbox[2], self.bbox[1]],
                    [self.bbox[2], self.bbox[3]],
                    [self.bbox[0], self.bbox[3]],
                    [self.bbox[0], self.bbox[1]]
                ]]
            }
        }

        m.add_geojson(strip_polygon, layer_name="شريط التحليل",
                      fill_colors=["blue"], fill_opacity=0.1,
                      stroke_colors=["blue"], stroke_widths=3)

        # إضافة الحقول
        with tempfile.NamedTemporaryFile(suffix='.geojson', delete=False) as tmp:
            temp_path = tmp.name
        gdf.to_file(temp_path, driver='GeoJSON')

        m.add_geojson(
            temp_path,
            layer_name=f"حدود الحقول ({stats['total_fields']} حقل - {stats['total_area_ha']:.1f} هكتار)",
            fill_colors=["green"],
            fill_opacity=0.3,
            stroke_colors=["red"],
            stroke_widths=2
        )

        from IPython.display import display
        display(m)

        os.unlink(temp_path)

    def plot_statistics(self):
        """رسم الإحصائيات"""
        if not self.results or not self.results['success']:
            print("لا توجد نتائج للرسم")
            return

        gdf = self.results['gdf']
        stats = self.results['stats']
        km_stats = stats['km_stats']

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('تحليل الحقول الزراعية - أسوان', fontsize=16, fontweight='bold')

        # 1. توزيع الحقول على طول الشريط
        ax1 = axes[0, 0]
        kms = [s['km'] for s in km_stats]
        counts = [s['num_fields'] for s in km_stats]
        bars1 = ax1.bar(kms, counts, color='green', alpha=0.7, edgecolor='darkgreen', linewidth=2)
        ax1.set_xlabel('المسافة (كم شرقًا)', fontsize=12)
        ax1.set_ylabel('عدد الحقول', fontsize=12)
        ax1.set_title('🌾 عدد الحقول لكل كيلومتر', fontsize=14, fontweight='bold')
        ax1.set_xticks(kms)
        ax1.grid(axis='y', alpha=0.3)

        for bar, count in zip(bars1, counts):
            if count > 0:
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        str(count), ha='center', va='bottom', fontweight='bold')

        # 2. المساحات على طول الشريط
        ax2 = axes[0, 1]
        areas = [s['area_ha'] for s in km_stats]
        bars2 = ax2.bar(kms, areas, color='orange', alpha=0.7, edgecolor='darkorange', linewidth=2)
        ax2.set_xlabel('المسافة (كم شرقًا)', fontsize=12)
        ax2.set_ylabel('المساحة (هكتار)', fontsize=12)
        ax2.set_title('📏 المساحة المزروعة لكل كيلومتر', fontsize=14, fontweight='bold')
        ax2.set_xticks(kms)
        ax2.grid(axis='y', alpha=0.3)

        for bar, area in zip(bars2, areas):
            if area > 0:
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        f'{area:.1f}', ha='center', va='bottom', fontweight='bold')

        # 3. توزيع أحجام الحقول
        ax3 = axes[1, 0]
        ax3.hist(gdf['area_ha'], bins=30, color='purple', alpha=0.7, edgecolor='black', linewidth=1.5)
        ax3.set_xlabel('مساحة الحقل (هكتار)', fontsize=12)
        ax3.set_ylabel('عدد الحقول', fontsize=12)
        ax3.set_title('📊 توزيع أحجام الحقول', fontsize=14, fontweight='bold')
        ax3.grid(axis='y', alpha=0.3)
        ax3.axvline(stats['mean_area_ha'], color='red', linestyle='--',
                    label=f'المتوسط: {stats["mean_area_ha"]:.2f} هكتار')
        ax3.legend()

        # 4. لوحة المعلومات
        ax4 = axes[1, 1]
        ax4.axis('off')

        info_text = f"""
        📊 ملخص النتائج النهائية
        {'='*40}

        🌾 إجمالي الحقول: {stats['total_fields']}
        📏 إجمالي المساحة: {stats['total_area_ha']:.2f} هكتار
        📐 متوسط المساحة: {stats['mean_area_ha']:.2f} هكتار
        📊 وسيط المساحة: {stats['median_area_ha']:.2f} هكتار
        🏞️ أكبر حقل: {stats['max_area_ha']:.2f} هكتار
        📏 أصغر حقل: {stats['min_area_ha']:.4f} هكتار
        📈 كثافة الحقول: {stats['density_per_km']:.1f} حقل/كم
        📐 المساحة لكل كم: {stats['area_per_km']:.2f} هكتار/كم
        """

        ax4.text(0.1, 0.5, info_text, transform=ax4.transAxes,
                fontsize=12, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

        plt.tight_layout()
        plt.show()

    def save_results(self, output_dir: str = "geoai_results"):
        """حفظ النتائج"""
        if not self.results or not self.results['success']:
            print("لا توجد نتائج للحفظ")
            return

        os.makedirs(output_dir, exist_ok=True)

        gdf = self.results['gdf']
        stats = self.results['stats']

        # حفظ GeoJSON
        geojson_path = os.path.join(output_dir, "fields.geojson")
        gdf.to_file(geojson_path, driver='GeoJSON')
        print(f"📁 تم حفظ: {geojson_path}")

        # حفظ KML
        kml_path = os.path.join(output_dir, "fields.kml")
        gdf.to_file(kml_path, driver='KML')
        print(f"📁 تم حفظ: {kml_path}")

        # حفظ الإحصائيات
        stats_df = pd.DataFrame(stats['km_stats'])
        stats_path = os.path.join(output_dir, "statistics.csv")
        stats_df.to_csv(stats_path, index=False, encoding='utf-8-sig')
        print(f"📁 تم حفظ: {stats_path}")

        # حفظ التقرير
        report_path = os.path.join(output_dir, "report.txt")
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("تقرير تحليل الحقول الزراعية\n")
            f.write("=" * 50 + "\n\n")
            f.write(f"إجمالي الحقول: {stats['total_fields']}\n")
            f.write(f"إجمالي المساحة: {stats['total_area_ha']:.2f} هكتار\n")
            f.write(f"متوسط المساحة: {stats['mean_area_ha']:.2f} هكتار\n")
            f.write(f"كثافة الحقول: {stats['density_per_km']:.1f} حقل/كم\n")

        print(f"📁 تم حفظ: {report_path}")


# ============================================================
# تنفيذ التحليل
# ============================================================

def main():
    """الدالة الرئيسية"""
    # إعدادات التحليل
    config = Config(
        start_lat=24.390288,
        start_lon=32.979508,
        strip_length_km=10,
        strip_width_km=1,
        zoom_level=18,
        overlap_ratio=0.15,
        num_segments=8,
        model_type="vit_l",
        min_field_area_ha=0.005,
        iou_threshold=0.4
    )

    # إنشاء المحلل وتنفيذ التحليل
    analyzer = AgriculturalStripAnalyzer(config)
    results = analyzer.run()

    if results['success']:
        # عرض النتائج
        analyzer.display_results()
        analyzer.plot_statistics()
        analyzer.save_results()

        # طباعة الملخص
        print("\n" + "=" * 60)
        print("🎉 اكتملت المعالجة بنجاح!")
        print("=" * 60)
        print(f"\n📊 النتائج النهائية:")
        print(f"   🌾 إجمالي الحقول: {results['stats']['total_fields']}")
        print(f"   📏 إجمالي المساحة: {results['stats']['total_area_ha']:.2f} هكتار")
        print(f"   📐 كثافة الحقول: {results['stats']['density_per_km']:.1f} حقل/كم")
        print("=" * 60)
    else:
        print("\n❌ فشل التحليل")


# تنفيذ الكود
if __name__ == "__main__":
    main()